# 08 Normalisation Strategies 2: BEN-style File-Grouped Training

This notebook contains the BEN-inspired normalisation experiment used in the thesis.

The goal is to test whether a **file-grouped training strategy**, inspired by Batch Effects Normalization (BEN), improves cross-file generalisation. The evaluation is performed at the **file level**, not at the cell level: complete microscopy files are held out for testing, and the model is trained on cells from the remaining files.

Important: this notebook is **inspired by BEN** rather than a full reproduction of the original BEN paper. The main idea tested here is whether grouping batches by file/acquisition context can reduce the impact of batch effects in this dataset.

## Experimental logic

The static CNN models showed very high performance when train and test data came from the same acquisition context, but poor performance when evaluated on independent files. BEN-style training is tested as a possible way to reduce this acquisition-specific shortcut.

Two representative timepoints are evaluated:

- **t = 0**, the initial static state of the acquisition.
- **t = 14**, the last timepoint shared by the original 15-timepoint files.

Testing both timepoints helps check whether failure at t = 0 is simply due to the initial timepoint being uninformative, or whether the same cross-file generalisation problem persists later in the acquisition.

In [ ]:
# ============================================================
# Imports
# ============================================================

from __future__ import annotations

import os
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Sampler

from sklearn.metrics import accuracy_score, recall_score, confusion_matrix, roc_auc_score
from sklearn.preprocessing import LabelEncoder

import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
# ============================================================
# Paths and configuration
# ============================================================

# Adjust these paths to your project structure.
PROJECT_ROOT = Path(".").resolve()
DATA_DIR = PROJECT_ROOT / "data" / "processed"        # expected: .npy crops and .csv metadata
RESULTS_DIR = PROJECT_ROOT / "results" / "normalisation" / "ben"
FIGURES_DIR = PROJECT_ROOT / "figures" / "normalisation" / "ben"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Expected crop shape: N x 7 x 512 x 512
N_CHANNELS = 7
N_CLASSES = 2

# BEN training parameters
BATCH_SIZE = 256
LR = 5e-4
EPOCHS = 3
WEIGHT_DECAY = 0.0

# Set to True if you want to use only a small number of cells per file while debugging.
DEBUG_SUBSAMPLE = False
DEBUG_N_PER_FILE = 200

## File manifest

The manifest maps the simplified file labels used in the thesis to their biological group and acquisition session.

The notebook assumes that, for each file, there is:

- one `.npy` array containing cell crops,
- one `.csv` metadata table with at least `time` and `cell_id` or equivalent fields.

If your filenames differ, edit the `crops_path` and `metadata_path` columns below.

In [ ]:
# ============================================================
# File manifest
# ============================================================

file_manifest = pd.DataFrame([
    {
        "file_label": "CNTL-MB231",
        "group": "Control",
        "session": 1,
        "crops_path": DATA_DIR / "CNTL-MB231_crops.npy",
        "metadata_path": DATA_DIR / "CNTL-MB231_metadata.csv",
    },
    {
        "file_label": "TAMO-MB231",
        "group": "Chemoresistant",
        "session": 1,
        "crops_path": DATA_DIR / "TAMO-MB231_crops.npy",
        "metadata_path": DATA_DIR / "TAMO-MB231_metadata.csv",
    },
    {
        "file_label": "CNTL_75uM_p1",
        "group": "Control",
        "session": 2,
        "crops_path": DATA_DIR / "CNTL_75uM_p1_crops.npy",
        "metadata_path": DATA_DIR / "CNTL_75uM_p1_metadata.csv",
    },
    {
        "file_label": "CNTL_75uM_p2",
        "group": "Control",
        "session": 2,
        "crops_path": DATA_DIR / "CNTL_75uM_p2_crops.npy",
        "metadata_path": DATA_DIR / "CNTL_75uM_p2_metadata.csv",
    },
    {
        "file_label": "CNTL_75uM_p3",
        "group": "Control",
        "session": 2,
        "crops_path": DATA_DIR / "CNTL_75uM_p3_crops.npy",
        "metadata_path": DATA_DIR / "CNTL_75uM_p3_metadata.csv",
    },
    {
        "file_label": "CNTL_75uM_p4",
        "group": "Control",
        "session": 2,
        "crops_path": DATA_DIR / "CNTL_75uM_p4_crops.npy",
        "metadata_path": DATA_DIR / "CNTL_75uM_p4_metadata.csv",
    },
    {
        "file_label": "TAMO_p1",
        "group": "Chemoresistant",
        "session": 3,
        "crops_path": DATA_DIR / "TAMO_p1_crops.npy",
        "metadata_path": DATA_DIR / "TAMO_p1_metadata.csv",
    },
    {
        "file_label": "TAMO_p2",
        "group": "Chemoresistant",
        "session": 3,
        "crops_path": DATA_DIR / "TAMO_p2_crops.npy",
        "metadata_path": DATA_DIR / "TAMO_p2_metadata.csv",
    },
])

file_manifest

## File-level validation scheme

Each fold holds out complete files. The held-out test set contains at least one control file and one chemoresistant file, so that binary metrics such as accuracy and AUROC can be interpreted more meaningfully than in a leave-one-file-out setting.

Edit the folds below if your final thesis uses a different exact grouping.

In [ ]:
# ============================================================
# File-level validation folds
# ============================================================

FOLDS = [
    {
        "fold": 1,
        "test_files": ["CNTL-MB231", "TAMO-MB231"],
    },
    {
        "fold": 2,
        "test_files": ["CNTL_75uM_p1", "CNTL_75uM_p2", "TAMO_p1"],
    },
    {
        "fold": 3,
        "test_files": ["CNTL_75uM_p3", "CNTL_75uM_p4", "TAMO_p2"],
    },
]

all_files = set(file_manifest["file_label"])
for fold in FOLDS:
    fold["train_files"] = sorted(list(all_files - set(fold["test_files"])))

pd.DataFrame(FOLDS)

In [ ]:
# ============================================================
# Data loading helpers
# ============================================================

def load_file_crops_and_metadata(row: pd.Series, timepoint: int) -> Tuple[np.ndarray, pd.DataFrame]:
    '''Load crops and metadata for one file and filter to a given timepoint.

    Expected:
    - crops: numpy array with shape N x 7 x H x W
    - metadata: CSV with one row per crop and a `time` column

    Adjust this function if your metadata uses a different time column name.
    '''
    crops_path = Path(row["crops_path"])
    meta_path = Path(row["metadata_path"])

    if not crops_path.exists():
        raise FileNotFoundError(f"Missing crops file: {crops_path}")
    if not meta_path.exists():
        raise FileNotFoundError(f"Missing metadata file: {meta_path}")

    crops = np.load(crops_path, mmap_mode="r")
    meta = pd.read_csv(meta_path)

    # Try common names for the timepoint column.
    time_col_candidates = ["time", "timepoint", "t", "tp"]
    time_col = next((c for c in time_col_candidates if c in meta.columns), None)
    if time_col is None:
        raise ValueError(f"No timepoint column found in {meta_path}. Available columns: {meta.columns.tolist()}")

    idx = np.where(meta[time_col].values == timepoint)[0]
    if len(idx) == 0:
        raise ValueError(f"No cells found for timepoint {timepoint} in {row['file_label']}")

    crops_tp = np.asarray(crops[idx])
    meta_tp = meta.iloc[idx].copy().reset_index(drop=True)
    meta_tp["file_label"] = row["file_label"]
    meta_tp["group"] = row["group"]
    meta_tp["session"] = row["session"]

    if DEBUG_SUBSAMPLE and len(meta_tp) > DEBUG_N_PER_FILE:
        rng = np.random.default_rng(SEED)
        keep = rng.choice(len(meta_tp), size=DEBUG_N_PER_FILE, replace=False)
        crops_tp = crops_tp[keep]
        meta_tp = meta_tp.iloc[keep].reset_index(drop=True)

    return crops_tp, meta_tp


def build_timepoint_dataset(timepoint: int) -> Tuple[np.ndarray, pd.DataFrame]:
    '''Load all files for one timepoint.'''
    all_crops = []
    all_meta = []

    for _, row in file_manifest.iterrows():
        crops, meta = load_file_crops_and_metadata(row, timepoint=timepoint)
        all_crops.append(crops)
        all_meta.append(meta)

    X = np.concatenate(all_crops, axis=0)
    meta = pd.concat(all_meta, ignore_index=True)
    y = (meta["group"].values == "Chemoresistant").astype(int)
    meta["label"] = y

    print(f"Loaded timepoint {timepoint}: X={X.shape}, metadata={meta.shape}")
    print(meta.groupby(["file_label", "group"]).size())
    return X, meta

In [ ]:
# ============================================================
# Dataset and file-grouped sampler
# ============================================================

class CellCropDataset(Dataset):
    def __init__(self, X: np.ndarray, meta: pd.DataFrame):
        self.X = X
        self.meta = meta.reset_index(drop=True)
        self.y = self.meta["label"].values.astype(np.int64)
        self.files = self.meta["file_label"].values

    def __len__(self):
        return len(self.meta)

    def __getitem__(self, idx):
        x = torch.tensor(self.X[idx], dtype=torch.float32)
        y = torch.tensor(self.y[idx], dtype=torch.long)
        file_label = self.files[idx]
        return x, y, file_label


class FileGroupedBatchSampler(Sampler[List[int]]):
    '''Sampler that creates mini-batches from one file at a time.

    This is the BEN-inspired component: each batch contains cells from the same
    microscopy file/acquisition context.
    '''
    def __init__(self, files: np.ndarray, batch_size: int, shuffle: bool = True, seed: int = 42):
        self.files = np.asarray(files)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.seed = seed
        self.rng = np.random.default_rng(seed)

        self.file_to_indices = {
            f: np.where(self.files == f)[0].tolist()
            for f in np.unique(self.files)
        }

    def __iter__(self):
        file_order = list(self.file_to_indices.keys())
        if self.shuffle:
            self.rng.shuffle(file_order)

        batches = []
        for f in file_order:
            indices = self.file_to_indices[f].copy()
            if self.shuffle:
                self.rng.shuffle(indices)
            for start in range(0, len(indices), self.batch_size):
                batch = indices[start:start + self.batch_size]
                if len(batch) > 0:
                    batches.append(batch)

        if self.shuffle:
            self.rng.shuffle(batches)

        for batch in batches:
            yield batch

    def __len__(self):
        total = 0
        for idxs in self.file_to_indices.values():
            total += int(np.ceil(len(idxs) / self.batch_size))
        return total

In [ ]:
# ============================================================
# Model
# ============================================================

class CompactCNN(nn.Module):
    def __init__(self, in_channels: int = 7, n_classes: int = 2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Linear(32, n_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

In [ ]:
# ============================================================
# Training and evaluation
# ============================================================

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    total = 0
    correct = 0

    for x, y, _ in loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item() * len(y)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += len(y)

    return {"loss": total_loss / total, "acc": correct / total}


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    y_true = []
    y_pred = []
    y_score = []
    files = []

    for x, y, file_label in loader:
        x = x.to(device)
        logits = model(x)
        probs = torch.softmax(logits, dim=1)[:, 1]
        pred = logits.argmax(dim=1)

        y_true.extend(y.numpy().tolist())
        y_pred.extend(pred.cpu().numpy().tolist())
        y_score.extend(probs.cpu().numpy().tolist())
        files.extend(list(file_label))

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    y_score = np.asarray(y_score)

    out = {
        "accuracy": accuracy_score(y_true, y_pred),
        "control_recall": recall_score(y_true, y_pred, pos_label=0, zero_division=0),
        "chemoresistant_recall": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "pred_control_pct": float(np.mean(y_pred == 0) * 100),
        "pred_chemoresistant_pct": float(np.mean(y_pred == 1) * 100),
        "n_test": len(y_true),
        "n_control": int(np.sum(y_true == 0)),
        "n_chemoresistant": int(np.sum(y_true == 1)),
    }

    if len(np.unique(y_true)) == 2:
        try:
            out["auroc"] = roc_auc_score(y_true, y_score)
        except ValueError:
            out["auroc"] = np.nan
    else:
        out["auroc"] = np.nan

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    return out, cm, pd.DataFrame({
        "file_label": files,
        "y_true": y_true,
        "y_pred": y_pred,
        "score_chemoresistant": y_score,
    })

In [ ]:
# ============================================================
# One fold runner
# ============================================================

def run_ben_fold(X: np.ndarray, meta: pd.DataFrame, fold_cfg: dict, timepoint: int):
    train_files = fold_cfg["train_files"]
    test_files = fold_cfg["test_files"]
    fold_id = fold_cfg["fold"]

    train_mask = meta["file_label"].isin(train_files).values
    test_mask = meta["file_label"].isin(test_files).values

    X_train = X[train_mask]
    meta_train = meta.loc[train_mask].reset_index(drop=True)
    X_test = X[test_mask]
    meta_test = meta.loc[test_mask].reset_index(drop=True)

    train_ds = CellCropDataset(X_train, meta_train)
    test_ds = CellCropDataset(X_test, meta_test)

    train_sampler = FileGroupedBatchSampler(
        files=meta_train["file_label"].values,
        batch_size=BATCH_SIZE,
        shuffle=True,
        seed=SEED + fold_id,
    )
    train_loader = DataLoader(train_ds, batch_sampler=train_sampler, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model = CompactCNN(in_channels=N_CHANNELS, n_classes=N_CLASSES).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()

    history = []
    for epoch in range(1, EPOCHS + 1):
        train_metrics = train_one_epoch(model, train_loader, optimizer, criterion)
        eval_metrics, _, _ = evaluate(model, test_loader)
        history.append({
            "fold": fold_id,
            "timepoint": timepoint,
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_acc": train_metrics["acc"],
            "test_acc": eval_metrics["accuracy"],
            "test_auroc": eval_metrics["auroc"],
            "pred_control_pct": eval_metrics["pred_control_pct"],
            "pred_chemoresistant_pct": eval_metrics["pred_chemoresistant_pct"],
        })

    final_metrics, cm, predictions = evaluate(model, test_loader)
    final_row = {
        "fold": fold_id,
        "timepoint": timepoint,
        "train_files": ", ".join(train_files),
        "test_files": ", ".join(test_files),
        **final_metrics,
    }

    predictions["fold"] = fold_id
    predictions["timepoint"] = timepoint

    return final_row, pd.DataFrame(history), predictions, cm

In [ ]:
# ============================================================
# Full BEN experiment for one timepoint
# ============================================================

def run_ben_experiment(timepoint: int):
    X, meta = build_timepoint_dataset(timepoint=timepoint)

    fold_results = []
    histories = []
    all_predictions = []
    confusion_matrices = {}

    for fold_cfg in FOLDS:
        print(f"\nRunning timepoint {timepoint}, fold {fold_cfg['fold']}")
        print("  Train:", fold_cfg["train_files"])
        print("  Test :", fold_cfg["test_files"])

        row, hist, pred, cm = run_ben_fold(X, meta, fold_cfg, timepoint=timepoint)
        fold_results.append(row)
        histories.append(hist)
        all_predictions.append(pred)
        confusion_matrices[fold_cfg["fold"]] = cm

        print("  Accuracy:", row["accuracy"])
        print("  AUROC:", row["auroc"])
        print("  Predicted control %:", row["pred_control_pct"])
        print("  Predicted chemoresistant %:", row["pred_chemoresistant_pct"])
        print("  CM [[control, tamo predicted cols]]:\n", cm)

    results_df = pd.DataFrame(fold_results)
    history_df = pd.concat(histories, ignore_index=True)
    predictions_df = pd.concat(all_predictions, ignore_index=True)

    # Save outputs
    results_df.to_csv(RESULTS_DIR / f"ben_t{timepoint}_fold_results.csv", index=False)
    history_df.to_csv(RESULTS_DIR / f"ben_t{timepoint}_training_history.csv", index=False)
    predictions_df.to_csv(RESULTS_DIR / f"ben_t{timepoint}_predictions.csv", index=False)

    return results_df, history_df, predictions_df, confusion_matrices

## Run BEN at t = 0

Run this cell when the real processed data are available. If you only want to test the notebook structure, set `DEBUG_SUBSAMPLE = True` above.

In [ ]:
# Uncomment to run with real data
# results_t0, history_t0, predictions_t0, cms_t0 = run_ben_experiment(timepoint=0)
# results_t0

## Run BEN at t = 14

This checks whether the same behaviour appears at a later static timepoint.

In [ ]:
# Uncomment to run with real data
# results_t14, history_t14, predictions_t14, cms_t14 = run_ben_experiment(timepoint=14)
# results_t14

## Mock results for thesis table

Use this only as a placeholder while writing. Replace with the real `results_t0` and `results_t14` outputs after rerunning the experiments.

In [ ]:
# ============================================================
# Mock BEN summary results
# Replace these values with the real outputs.
# ============================================================

mock_ben_summary = pd.DataFrame([
    {
        "timepoint": 0,
        "accuracy_mean": 0.512,
        "accuracy_std": 0.024,
        "pred_control_pct_mean": 100.0,
        "pred_control_pct_std": 0.0,
        "pred_chemoresistant_pct_mean": 0.0,
        "pred_chemoresistant_pct_std": 0.0,
        "control_recall_mean": 1.0,
        "control_recall_std": 0.0,
        "chemoresistant_recall_mean": 0.0,
        "chemoresistant_recall_std": 0.0,
        "interpretation": "single-class collapse to control",
    },
    {
        "timepoint": 14,
        "accuracy_mean": 0.508,
        "accuracy_std": 0.021,
        "pred_control_pct_mean": 100.0,
        "pred_control_pct_std": 0.0,
        "pred_chemoresistant_pct_mean": 0.0,
        "pred_chemoresistant_pct_std": 0.0,
        "control_recall_mean": 1.0,
        "control_recall_std": 0.0,
        "chemoresistant_recall_mean": 0.0,
        "chemoresistant_recall_std": 0.0,
        "interpretation": "same failure mode at later timepoint",
    },
])

mock_ben_summary.to_csv(RESULTS_DIR / "ben_summary_mock.csv", index=False)
mock_ben_summary

In [ ]:
# ============================================================
# Helper: summarise real results when available
# ============================================================

def summarise_ben_results(results_df: pd.DataFrame, timepoint: int) -> dict:
    return {
        "timepoint": timepoint,
        "accuracy_mean": results_df["accuracy"].mean(),
        "accuracy_std": results_df["accuracy"].std(ddof=1),
        "auroc_mean": results_df["auroc"].mean(skipna=True),
        "auroc_std": results_df["auroc"].std(skipna=True, ddof=1),
        "pred_control_pct_mean": results_df["pred_control_pct"].mean(),
        "pred_control_pct_std": results_df["pred_control_pct"].std(ddof=1),
        "pred_chemoresistant_pct_mean": results_df["pred_chemoresistant_pct"].mean(),
        "pred_chemoresistant_pct_std": results_df["pred_chemoresistant_pct"].std(ddof=1),
        "control_recall_mean": results_df["control_recall"].mean(),
        "control_recall_std": results_df["control_recall"].std(ddof=1),
        "chemoresistant_recall_mean": results_df["chemoresistant_recall"].mean(),
        "chemoresistant_recall_std": results_df["chemoresistant_recall"].std(ddof=1),
    }

# Example after running real experiments:
# ben_summary = pd.DataFrame([
#     summarise_ben_results(results_t0, timepoint=0),
#     summarise_ben_results(results_t14, timepoint=14),
# ])
# ben_summary.to_csv(RESULTS_DIR / "ben_summary_real.csv", index=False)
# ben_summary

In [ ]:
# ============================================================
# Thesis-style formatting
# ============================================================

def fmt_pct(mean, std=None, decimals=1):
    if std is None or pd.isna(std):
        return f"{mean * 100:.{decimals}f}\\%"
    return f"${mean * 100:.{decimals}f} \\pm {std * 100:.{decimals}f}\\%$"


def make_ben_thesis_table(summary_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, r in summary_df.iterrows():
        rows.append({
            "Timepoint": f"$t={int(r['timepoint'])}$",
            "Accuracy": fmt_pct(r["accuracy_mean"], r["accuracy_std"]),
            "Predicted Control": f"${r['pred_control_pct_mean']:.1f} \\pm {r['pred_control_pct_std']:.1f}\\%$",
            "Predicted TAMO": f"${r['pred_chemoresistant_pct_mean']:.1f} \\pm {r['pred_chemoresistant_pct_std']:.1f}\\%$",
            "Control Recall": fmt_pct(r["control_recall_mean"], r["control_recall_std"]),
            "TAMO Recall": fmt_pct(r["chemoresistant_recall_mean"], r["chemoresistant_recall_std"]),
        })
    return pd.DataFrame(rows)

thesis_ben_table = make_ben_thesis_table(mock_ben_summary)
thesis_ben_table

In [ ]:
latex = thesis_ben_table.to_latex(
    index=False,
    escape=False,
    caption="BEN performance under file-level grouped validation at representative timepoints. Values shown here are placeholders and should be replaced with final experimental results.",
    label="tab:ben_summary",
)

with open(RESULTS_DIR / "ben_summary_mock.tex", "w") as f:
    f.write(latex)

print(latex)